In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(r"C:\Users\luis\Documents\TFG - Data-Centric AI")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.TemporaryClean import add_temporal_groups, calculate_similarity
from utils.thresholds.deduplication import (
    evaluate_threshold_grid,
    sample_pairs_for_threshold_combination_review,
    export_pair_review_images,
    evaluate_thresholds_against_manual_labels,
    recommend_conservative_threshold,
)

# Centralized paths used by the threshold tuning workflow.
METADATA_PATH = PROJECT_ROOT / "data" / "phase2" / "phase2_train.csv"
IMAGES_DIR = PROJECT_ROOT / "data" / "phase2" / "frames"

REPORTS_DIR = PROJECT_ROOT / "reports" / "deduplication_threshold_selection"
PAIR_REVIEW_DIR = REPORTS_DIR / "pair_review_images"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PAIR_REVIEW_DIR.mkdir(parents=True, exist_ok=True)

# Candidate pairs are only compared within short temporal windows from the same video.
TEMPORAL_TOLERANCE_SECONDS = 6.0

# Candidate grid generated from the initial threshold pair.
INITIAL_SSIM_THRESHOLD = 0.80
INITIAL_PHASH_DISTANCE_THRESHOLD = 10
SSIM_FLUCTUATIONS = [-0.10, -0.05, 0.00, 0.05, 0.10]
PHASH_FLUCTUATIONS = [-4, -2, 0, 2, 4]
SSIM_CANDIDATES = [round(INITIAL_SSIM_THRESHOLD + delta, 2) for delta in SSIM_FLUCTUATIONS]
PHASH_CANDIDATES = [INITIAL_PHASH_DISTANCE_THRESHOLD + delta for delta in PHASH_FLUCTUATIONS]
N_REVIEW_PAIRS = 100
N_REVIEW_PAIRS_PER_COMBINATION = N_REVIEW_PAIRS // (len(SSIM_CANDIDATES) * len(PHASH_CANDIDATES))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("METADATA_PATH:", METADATA_PATH)
print("IMAGES_DIR:", IMAGES_DIR)

In [ ]:
df = pd.read_csv(METADATA_PATH)

print("Training images:", len(df))
display(df.head())

# Assign a temporal group id so only nearby frames are compared with each other.
grouped_df = add_temporal_groups(
    dataframe=df,
    temporal_tolerance_seconds=TEMPORAL_TOLERANCE_SECONDS,
)

# Summarize each group before deriving the total number of pairwise comparisons.
temporal_summary = (
    grouped_df
    .groupby("group_id")
    .agg(
        num_images=("filename", "count"),
        histology=("histology", "first"),
        min_time=("elapsed_seconds", "min"),
        max_time=("elapsed_seconds", "max"),
    )
    .reset_index()
)

temporal_summary["duration"] = (
    temporal_summary["max_time"] - temporal_summary["min_time"]
)

# Number of unordered image pairs inside every temporal group: n * (n - 1) / 2.
total_pairs = int(
    temporal_summary["num_images"]
    .apply(lambda n: n * (n - 1) / 2)
    .sum()
)

print("Temporal groups:", grouped_df["group_id"].nunique())
print("Comparable pairs:", total_pairs)

In [ ]:
similarity_pairs_df = calculate_similarity(
    dataframe=grouped_df,
    images_dir=IMAGES_DIR,
    ssim_threshold=INITIAL_SSIM_THRESHOLD,
    phash_distance_threshold=INITIAL_PHASH_DISTANCE_THRESHOLD,
)

print("Calculated comparable pairs:", len(similarity_pairs_df))

display(similarity_pairs_df.head())
display(similarity_pairs_df[["ssim", "phash_distance"]].describe())

In [ ]:
# Evaluate each SSIM/pHash combination over the same precomputed pair table.
threshold_grid_df = evaluate_threshold_grid(
    similarity_pairs_df=similarity_pairs_df,
    ssim_thresholds=SSIM_CANDIDATES,
    phash_distance_thresholds=PHASH_CANDIDATES,
)

# This percentage is an aggressiveness proxy: higher values remove more pairs.
threshold_grid_df["redundant_pair_percentage"] = (
    100 * threshold_grid_df["redundant_pairs"] / threshold_grid_df["total_pairs"]
)

threshold_grid_df = threshold_grid_df.sort_values(
    ["ssim_threshold", "phash_distance_threshold"],
    ascending=[False, True],
).reset_index(drop=True)

# Heatmap matrix: rows are SSIM thresholds and columns are pHash distance thresholds.
pivot = threshold_grid_df.pivot(
    index="ssim_threshold",
    columns="phash_distance_threshold",
    values="redundant_pair_percentage",
)

plt.figure(figsize=(8, 5))
plt.imshow(pivot.values, aspect="auto")
plt.xticks(range(len(pivot.columns)), pivot.columns)
plt.yticks(range(len(pivot.index)), pivot.index)
plt.xlabel("pHash distance threshold")
plt.ylabel("SSIM threshold")
plt.title("% of pairs marked as redundant")
plt.colorbar(label="% redundant pairs")
plt.show()

In [ ]:
# Sample candidate pairs around every SSIM/pHash threshold combination.
review_pairs_df = sample_pairs_for_threshold_combination_review(
    similarity_pairs_df=similarity_pairs_df,
    ssim_thresholds=SSIM_CANDIDATES,
    phash_distance_thresholds=PHASH_CANDIDATES,
    n_per_combination=N_REVIEW_PAIRS_PER_COMBINATION,
)

# Export side-by-side images so labels can be assigned outside the notebook.
review_pairs_df = export_pair_review_images(
    review_pairs_df=review_pairs_df,
    images_dir=IMAGES_DIR,
    output_dir=PAIR_REVIEW_DIR,
    image_size=(320, 180),
)

REVIEW_CSV_PATH = REPORTS_DIR / "manual_pair_review.csv"
review_pairs_df.to_csv(REVIEW_CSV_PATH, index=False)

print("CSV to label:", REVIEW_CSV_PATH)
print("Images exported to:", PAIR_REVIEW_DIR)

In [ ]:
labeled_pairs_df = pd.read_csv(REVIEW_CSV_PATH, sep=None, engine="python")

# Normalize manual labels before validating them against the accepted vocabulary.
labeled_pairs_df["manual_label"] = (
    labeled_pairs_df["manual_label"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

valid_labels = {
    "duplicated",
    "different",
    "uncertain",
}

invalid_labels = set(labeled_pairs_df["manual_label"].unique()) - valid_labels

if invalid_labels:
    raise ValueError(f"Invalid labels: {invalid_labels}")

label_counts = (
    labeled_pairs_df["manual_label"]
    .value_counts()
    .rename_axis("manual_label")
    .reset_index(name="count")
)

print(label_counts)

In [ ]:
# Treat duplicated pairs as positives and different pairs as negatives.
manual_evaluation_df = evaluate_thresholds_against_manual_labels(
    labeled_pairs_df=labeled_pairs_df,
    ssim_thresholds=SSIM_CANDIDATES,
    phash_distance_thresholds=PHASH_CANDIDATES,
    positive_labels=("duplicated",),
    negative_labels=("different",),
)

# Prefer high precision first, then recall, while minimizing false positives.
manual_evaluation_df = manual_evaluation_df.sort_values(
    ["precision", "recall", "false_positive"],
    ascending=[False, False, True],
).reset_index(drop=True)

display(manual_evaluation_df)

In [ ]:
# Select the most conservative threshold pair that satisfies the minimum precision target.
recommended_df = recommend_conservative_threshold(
    evaluation_df=manual_evaluation_df,
    min_precision=0.90,
)

print(recommended_df)

selected = recommended_df.iloc[0]

FINAL_SSIM_THRESHOLD = float(selected["ssim_threshold"])
FINAL_PHASH_DISTANCE_THRESHOLD = int(selected["phash_distance_threshold"])

print("Selected final thresholds:")
print(f"SSIM >= {FINAL_SSIM_THRESHOLD:.2f}")
print(f"pHash distance <= {FINAL_PHASH_DISTANCE_THRESHOLD}")

final_text = f"""
Selected configuration for the deduplication phase:

- Temporal window: {TEMPORAL_TOLERANCE_SECONDS:.1f} seconds
- SSIM threshold: {FINAL_SSIM_THRESHOLD:.2f}
- pHash distance threshold: {FINAL_PHASH_DISTANCE_THRESHOLD}

The redundancy decision requires both conditions to be met at the same time.
Thresholds were selected through visual review of candidate pairs, prioritizing a conservative strategy: maximize precision for pairs marked as duplicated to avoid removing images that add useful training variability.
"""

print(final_text)